In [1]:
ppid = 12008
shift = 1.242188034702849
starting_time = "2024-09-02 10:36:53.500"    # do a manual entry - enter as YYYY-MM-DD HH:MM:SS.fff   this is the start time of the trimmed resp files

# Supine

## 1. VUAMS Impedance Calibration (Supine)

In [2]:
import numpy as np

vuams_depths_supine = np.fromstring(
    "51.97405433 57.63725167 37.28014198 47.98313947 50.47542702 34.98117011 51.52623604 49.77502437 35.83683353",
    sep=" ",
)

tv_supine_spiro = np.fromstring(
    "1.20977533 1.75793892 1.41731778 1.84688199 1.55642605 1.39112541 1.26200905 1.02277975 1.29152939",
    sep=" ",
)

In [3]:
import numpy as np

# Ensure datasets have the same length
assert len(vuams_depths_supine) == len(tv_supine_spiro)

# Predictor matrix without intercept column (forced zero intercept)
X = vuams_depths_supine[:, np.newaxis]
y = tv_supine_spiro

# Pseudoinverse regression forced through origin (c = 0)
X_pinv = np.linalg.pinv(X)
vuams_slope_supine = np.dot(X_pinv, y)[0]

print(f"VUAMS Slope (Supine): {vuams_slope_supine:.8f}")

# Predicted Tidal Volume
tv_supine_vuams_converted = vuams_slope_supine * vuams_depths_supine

# Metrics: MAPE & RMSE
vuams_mape_supine = np.mean(np.abs((y - tv_supine_vuams_converted) / y)) * 100
vuams_rmse_supine = np.sqrt(np.mean((y - tv_supine_vuams_converted) ** 2))

print(f"VUAMS Supine MAPE: {vuams_mape_supine:.4f}%")
print(f"VUAMS Supine RMSE: {vuams_rmse_supine:.6f} L")

VUAMS Slope (Supine): 0.02989889
VUAMS Supine MAPE: 20.7300%
VUAMS Supine RMSE: 0.306467 L


## 2. Hexoskin 2-DOF Calibration: Thorax + Abdomen (Supine)

In [4]:
import numpy as np

hexoskin_thorax_depths_supine = np.fromstring(
    "42.2550405  25.82646079 19.92086017 24.11657575 34.27409905 14.19548267 24.07242201 23.84824449 20.24830162",
    sep=" ",
)

hexoskin_abdomen_depths_supine = np.fromstring(
    "39.40666191 41.83877246 26.19019124 35.36622474 37.61798096 25.62593797 36.07800602 35.18070006 37.27902213",
    sep=" ",
)

tv_supine_spiro = np.fromstring(
    "1.75793892 1.41731778 1.84688199 1.55642605 1.39112541 1.26200905 1.02277975 1.29152939 1.36358264",
    sep=" ",
)

In [5]:
import numpy as np

# Ensure datasets have the same length
assert len(tv_supine_spiro) == len(hexoskin_thorax_depths_supine) == len(hexoskin_abdomen_depths_supine)

# Predictors matrix without intercept column
X = np.column_stack((
    hexoskin_thorax_depths_supine,
    hexoskin_abdomen_depths_supine
))
y = tv_supine_spiro

# Pseudoinverse regression forced through origin (c = 0)
X_pinv = np.linalg.pinv(X)
twodegrees_slope_thorax_supine, twodegrees_slope_abdomen_supine = np.dot(X_pinv, y)

print(f"Slope Thorax: {twodegrees_slope_thorax_supine:.8f}")
print(f"Slope Abdomen: {twodegrees_slope_abdomen_supine:.8f}")

# Predicted Tidal Volume
tv_supine_hexoskin_2degrees_converted = (
    twodegrees_slope_thorax_supine * hexoskin_thorax_depths_supine +
    twodegrees_slope_abdomen_supine * hexoskin_abdomen_depths_supine
)

# Metrics: MAPE & RMSE
twodegrees_mape_supine = np.mean(np.abs((y - tv_supine_hexoskin_2degrees_converted) / y)) * 100
twodegrees_rmse_supine = np.sqrt(np.mean((y - tv_supine_hexoskin_2degrees_converted) ** 2))

print(f"Hexoskin 2-DOF Supine MAPE: {twodegrees_mape_supine:.4f}%")
print(f"Hexoskin 2-DOF Supine RMSE: {twodegrees_rmse_supine:.6f} L")

Slope Thorax: 0.01287806
Slope Abdomen: 0.03058203
Hexoskin 2-DOF Supine MAPE: 17.0417%
Hexoskin 2-DOF Supine RMSE: 0.329654 L


## 3. Hexoskin Thorax-Only Calibration (Supine)

In [6]:
import numpy as np

hexoskin_thorax_depths_supine = np.fromstring(
    "42.2550405  25.82646079 19.92086017 24.11657575 34.27409905 14.19548267 24.07242201 23.84824449 20.24830162",
    sep=" ",
)


tv_supine_spiro = np.fromstring(
    "1.75793892 1.41731778 1.84688199 1.55642605 1.39112541 1.26200905 1.02277975 1.29152939 1.36358264",
    sep=" ",
)

In [7]:
import numpy as np

# Ensure datasets have the same length
assert len(hexoskin_thorax_depths_supine) == len(tv_supine_spiro)

# Predictor matrix without intercept column
X = hexoskin_thorax_depths_supine[:, np.newaxis]
y = tv_supine_spiro

# Pseudoinverse regression forced through origin (c = 0)
X_pinv = np.linalg.pinv(X)
hexoskin_thorax_slope_supine = np.dot(X_pinv, y)[0]

print(f"Thorax-Only Slope (Supine): {hexoskin_thorax_slope_supine:.8f}")

# Predicted Tidal Volume
tv_supine_thorax_converted = hexoskin_thorax_slope_supine * hexoskin_thorax_depths_supine

# Metrics: MAPE & RMSE
hexoskin_thorax_mape_supine = np.mean(np.abs((y - tv_supine_thorax_converted) / y)) * 100
hexoskin_thorax_rmse_supine = np.sqrt(np.mean((y - tv_supine_thorax_converted) ** 2))

print(f"Thorax-Only Supine MAPE: {hexoskin_thorax_mape_supine:.4f}%")
print(f"Thorax-Only Supine RMSE: {hexoskin_thorax_rmse_supine:.6f} L")

Thorax-Only Slope (Supine): 0.05244607
Thorax-Only Supine MAPE: 23.5112%
Thorax-Only Supine RMSE: 0.411690 L


## 4. Hexoskin Abdomen-Only Calibration (Supine)

In [8]:
import numpy as np

hexoskin_abdomen_depths_supine = np.fromstring(
    "39.40666191 41.83877246 26.19019124 35.36622474 37.61798096 25.62593797 36.07800602 35.18070006 37.27902213",
    sep=" ",
)

tv_supine_spiro = np.fromstring(
    "1.75793892 1.41731778 1.84688199 1.55642605 1.39112541 1.26200905 1.02277975 1.29152939 1.36358264",
    sep=" ",
)

In [9]:
import numpy as np

# Ensure datasets have the same length
assert len(hexoskin_abdomen_depths_supine) == len(tv_supine_spiro)

# Predictor matrix without intercept column
X = hexoskin_abdomen_depths_supine[:, np.newaxis]
y = tv_supine_spiro

# Pseudoinverse regression forced through origin (c = 0)
X_pinv = np.linalg.pinv(X)
hexoskin_abdomen_slope_supine = np.dot(X_pinv, y)[0]

print(f"Abdomen-Only Slope (Supine): {hexoskin_abdomen_slope_supine:.8f}")

# Predicted Tidal Volume
tv_supine_abdomen_converted = hexoskin_abdomen_slope_supine * hexoskin_abdomen_depths_supine

# Metrics: MAPE & RMSE
hexoskin_abdomen_mape_supine = np.mean(np.abs((y - tv_supine_abdomen_converted) / y)) * 100
hexoskin_abdomen_rmse_supine = np.sqrt(np.mean((y - tv_supine_abdomen_converted) ** 2))

print(f"Abdomen-Only Supine MAPE: {hexoskin_abdomen_mape_supine:.4f}%")
print(f"Abdomen-Only Supine RMSE: {hexoskin_abdomen_rmse_supine:.6f} L")

Abdomen-Only Slope (Supine): 0.04001725
Abdomen-Only Supine MAPE: 18.5855%
Abdomen-Only Supine RMSE: 0.338789 L


# Sitting

## 1. VUAMS Impedance Calibration (Sitting)

In [10]:
import numpy as np

vuams_depths_sitting = np.fromstring(
    "36.3780929  24.73935261 23.09930999 27.74398154 33.47464672 30.58307317 38.70010764 61.48185974",
    sep=" ",
)

tv_sitting_spiro = np.fromstring(
    "1.0093745  1.62991658 1.21372396 1.23830032 1.39481544 1.27578878 1.42059994 2.34556323",
    sep=" ",
)

In [11]:
import numpy as np

# Ensure datasets have the same length
assert len(vuams_depths_sitting) == len(tv_sitting_spiro)

# Predictor matrix without intercept column
X = vuams_depths_sitting[:, np.newaxis]
y = tv_sitting_spiro

# Pseudoinverse regression forced through origin (c = 0)
X_pinv = np.linalg.pinv(X)
vuams_slope_sitting = np.dot(X_pinv, y)[0]

print(f"VUAMS Slope (Sitting): {vuams_slope_sitting:.8f}")

# Predicted Tidal Volume
tv_sitting_vuams_converted = vuams_slope_sitting * vuams_depths_sitting

# Metrics: MAPE & RMSE
vuams_mape_sitting = np.mean(np.abs((y - tv_sitting_vuams_converted) / y)) * 100
vuams_rmse_sitting = np.sqrt(np.mean((y - tv_sitting_vuams_converted) ** 2))

print(f"VUAMS Sitting MAPE: {vuams_mape_sitting:.4f}%")
print(f"VUAMS Sitting RMSE: {vuams_rmse_sitting:.6f} L")

VUAMS Slope (Sitting): 0.04013594
VUAMS Sitting MAPE: 17.4284%
VUAMS Sitting RMSE: 0.304968 L


## 2. Hexoskin 2-DOF Calibration: Thorax + Abdomen (Sitting)

In [12]:
import numpy as np

hexoskin_thorax_depths_sitting = np.fromstring(
    "42.59074773  39.89883112  32.54459155  37.13194091  35.15060725 45.34834473  58.63955329 134.73815801",
    sep=" ",
)

hexoskin_abdomen_depths_sitting = np.fromstring(
    "41.40153357 31.47982378 26.53048319 31.22466849 44.68479493 39.49203913 45.71356924 57.81523101",
    sep=" ",
)

tv_sitting_spiro = np.fromstring(
    "1.0093745  1.62991658 1.21372396 1.23830032 1.39481544 1.27578878 1.42059994 2.34556323",
    sep=" ",
)

In [13]:
import numpy as np

# Ensure datasets have the same length
assert len(tv_sitting_spiro) == len(hexoskin_thorax_depths_sitting) == len(hexoskin_abdomen_depths_sitting)

# Predictors matrix without intercept column
X = np.column_stack((
    hexoskin_thorax_depths_sitting,
    hexoskin_abdomen_depths_sitting
))
y = tv_sitting_spiro

# Pseudoinverse regression forced through origin (c = 0)
X_pinv = np.linalg.pinv(X)
twodegrees_slope_thorax_sitting, twodegrees_slope_abdomen_sitting = np.dot(X_pinv, y)

print(f"Slope Thorax: {twodegrees_slope_thorax_sitting:.8f}")
print(f"Slope Abdomen: {twodegrees_slope_abdomen_sitting:.8f}")

# Predicted Tidal Volume
tv_sitting_hexoskin_2degrees_converted = (
    twodegrees_slope_thorax_sitting * hexoskin_thorax_depths_sitting +
    twodegrees_slope_abdomen_sitting * hexoskin_abdomen_depths_sitting
)

# Metrics: MAPE & RMSE
twodegrees_mape_sitting = np.mean(np.abs((y - tv_sitting_hexoskin_2degrees_converted) / y)) * 100
twodegrees_rmse_sitting = np.sqrt(np.mean((y - tv_sitting_hexoskin_2degrees_converted) ** 2))

print(f"Hexoskin 2-DOF Sitting MAPE: {twodegrees_mape_sitting:.4f}%")
print(f"Hexoskin 2-DOF Sitting RMSE: {twodegrees_rmse_sitting:.6f} L")

Slope Thorax: 0.00639747
Slope Abdomen: 0.02665734
Hexoskin 2-DOF Sitting MAPE: 16.0986%
Hexoskin 2-DOF Sitting RMSE: 0.268525 L


## 3. Hexoskin Thorax-Only Calibration (Sitting)

In [14]:
import numpy as np

hexoskin_thorax_depths_sitting = np.fromstring(
    "42.59074773  39.89883112  32.54459155  37.13194091  35.15060725 45.34834473  58.63955329 134.73815801",
    sep=" ",
)


tv_sitting_spiro = np.fromstring(
    "1.0093745  1.62991658 1.21372396 1.23830032 1.39481544 1.27578878 1.42059994 2.34556323",
    sep=" ",
)

In [15]:
import numpy as np

# Ensure datasets have the same length
assert len(hexoskin_thorax_depths_sitting) == len(tv_sitting_spiro)

# Predictor matrix without intercept column
X = hexoskin_thorax_depths_sitting[:, np.newaxis]
y = tv_sitting_spiro

# Pseudoinverse regression forced through origin (c = 0)
X_pinv = np.linalg.pinv(X)
hexoskin_thorax_slope_sitting = np.dot(X_pinv, y)[0]

print(f"Thorax-Only Slope (Sitting): {hexoskin_thorax_slope_sitting:.8f}")

# Predicted Tidal Volume
tv_sitting_thorax_converted = hexoskin_thorax_slope_sitting * hexoskin_thorax_depths_sitting

# Metrics: MAPE & RMSE
hexoskin_thorax_mape_sitting = np.mean(np.abs((y - tv_sitting_thorax_converted) / y)) * 100
hexoskin_thorax_rmse_sitting = np.sqrt(np.mean((y - tv_sitting_thorax_converted) ** 2))

print(f"Thorax-Only Sitting MAPE: {hexoskin_thorax_mape_sitting:.4f}%")
print(f"Thorax-Only Sitting RMSE: {hexoskin_thorax_rmse_sitting:.6f} L")

Thorax-Only Slope (Sitting): 0.02277100
Thorax-Only Sitting MAPE: 27.1653%
Thorax-Only Sitting RMSE: 0.479770 L


## 4. Hexoskin Abdomen-Only Calibration (Sitting)

In [16]:
import numpy as np

hexoskin_abdomen_depths_sitting = np.fromstring(
    "41.40153357 31.47982378 26.53048319 31.22466849 44.68479493 39.49203913 45.71356924 57.81523101",
    sep=" ",
)

tv_sitting_spiro = np.fromstring(
    "1.0093745  1.62991658 1.21372396 1.23830032 1.39481544 1.27578878 1.42059994 2.34556323",
    sep=" ",
)

In [17]:
import numpy as np

# Ensure datasets have the same length
assert len(hexoskin_abdomen_depths_sitting) == len(tv_sitting_spiro)

# Predictor matrix without intercept column
X = hexoskin_abdomen_depths_sitting[:, np.newaxis]
y = tv_sitting_spiro

# Pseudoinverse regression forced through origin (c = 0)
X_pinv = np.linalg.pinv(X)
hexoskin_abdomen_slope_sitting = np.dot(X_pinv, y)[0]

print(f"Abdomen-Only Slope (Sitting): {hexoskin_abdomen_slope_sitting:.8f}")

# Predicted Tidal Volume
tv_sitting_abdomen_converted = hexoskin_abdomen_slope_sitting * hexoskin_abdomen_depths_sitting

# Metrics: MAPE & RMSE
hexoskin_abdomen_mape_sitting = np.mean(np.abs((y - tv_sitting_abdomen_converted) / y)) * 100
hexoskin_abdomen_rmse_sitting = np.sqrt(np.mean((y - tv_sitting_abdomen_converted) ** 2))

print(f"Abdomen-Only Sitting MAPE: {hexoskin_abdomen_mape_sitting:.4f}%")
print(f"Abdomen-Only Sitting RMSE: {hexoskin_abdomen_rmse_sitting:.6f} L")

Abdomen-Only Slope (Sitting): 0.03568685
Abdomen-Only Sitting MAPE: 20.1409%
Abdomen-Only Sitting RMSE: 0.305002 L


# Standing posture

## 1. VUAMS Impedance Calibration (Standing)

In [18]:
import numpy as np

vuams_depths_standing = np.fromstring(
    "57.83041003 64.73942104 67.18394174 60.28071598 65.07205775 75.51072656 68.30793261 55.18753057 73.05555716 72.9576902",
    sep=" ",
)

tv_standing_spiro = np.fromstring(
    "2.14765981 2.59810108 2.04987985 2.12560338 2.07429475 2.02826924 1.90354675 1.66025174 1.82820472 2.14254895",
    sep=" ",
)

In [19]:
import numpy as np

# Ensure datasets have the same length
assert len(vuams_depths_standing) == len(tv_standing_spiro)

# Predictor matrix without intercept column
X = vuams_depths_standing[:, np.newaxis]
y = tv_standing_spiro

# Pseudoinverse regression forced through origin (c = 0)
X_pinv = np.linalg.pinv(X)
vuams_slope_standing = np.dot(X_pinv, y)[0]

print(f"VUAMS Slope (Standing): {vuams_slope_standing:.8f}")

# Predicted Tidal Volume
tv_standing_vuams_converted = vuams_slope_standing * vuams_depths_standing

# Metrics: MAPE & RMSE
vuams_mape_standing = np.mean(np.abs((y - tv_standing_vuams_converted) / y)) * 100
vuams_rmse_standing = np.sqrt(np.mean((y - tv_standing_vuams_converted) ** 2))

print(f"VUAMS Standing MAPE: {vuams_mape_standing:.4f}%")
print(f"VUAMS Standing RMSE: {vuams_rmse_standing:.6f} L")

VUAMS Slope (Standing): 0.03086814
VUAMS Standing MAPE: 11.3514%
VUAMS Standing RMSE: 0.299302 L


## 2. Hexoskin 2-DOF Calibration: Thorax + Abdomen (Standing)

In [20]:
import numpy as np

hexoskin_thorax_depths_standing = np.fromstring(
    "116.89948604 125.76852837 147.47609114 115.36355417 121.32316084 115.34567963 127.7271245  108.72384895 119.25775732 113.20585803",
    sep=" ",
)

hexoskin_abdomen_depths_standing = np.fromstring(
    "42.34629886 44.48337263 53.79494177 50.2668109  51.21436216 53.29261165 49.76386574 41.48242151 53.11484646 56.48568042",
    sep=" ",
)

tv_standing_spiro = np.fromstring(
    "2.14765981 2.59810108 2.04987985 2.12560338 2.07429475 2.02826924 1.90354675 1.66025174 1.82820472 2.14254895",
    sep=" ",
)

In [21]:
import numpy as np

# Ensure datasets have the same length
assert len(tv_standing_spiro) == len(hexoskin_thorax_depths_standing) == len(hexoskin_abdomen_depths_standing)

# Predictors matrix without intercept column
X = np.column_stack((
    hexoskin_thorax_depths_standing,
    hexoskin_abdomen_depths_standing
))
y = tv_standing_spiro

# Pseudoinverse regression forced through origin (c = 0)
X_pinv = np.linalg.pinv(X)
twodegrees_slope_thorax_standing, twodegrees_slope_abdomen_standing = np.dot(X_pinv, y)

print(f"Slope Thorax: {twodegrees_slope_thorax_standing:.8f}")
print(f"Slope Abdomen: {twodegrees_slope_abdomen_standing:.8f}")

# Predicted Tidal Volume
tv_standing_hexoskin_2degrees_converted = (
    twodegrees_slope_thorax_standing * hexoskin_thorax_depths_standing +
    twodegrees_slope_abdomen_standing * hexoskin_abdomen_depths_standing
)

# Metrics: MAPE & RMSE
twodegrees_mape_standing = np.mean(np.abs((y - tv_standing_hexoskin_2degrees_converted) / y)) * 100
twodegrees_rmse_standing = np.sqrt(np.mean((y - tv_standing_hexoskin_2degrees_converted) ** 2))

print(f"Hexoskin 2-DOF Standing MAPE: {twodegrees_mape_standing:.4f}%")
print(f"Hexoskin 2-DOF Standing RMSE: {twodegrees_rmse_standing:.6f} L")

Slope Thorax: 0.01370315
Slope Abdomen: 0.00781463
Hexoskin 2-DOF Standing MAPE: 9.9354%
Hexoskin 2-DOF Standing RMSE: 0.255307 L


## 3. Hexoskin Thorax-Only Calibration (Standing)

In [22]:
import numpy as np

hexoskin_thorax_depths_standing = np.fromstring(
    "116.89948604 125.76852837 147.47609114 115.36355417 121.32316084 115.34567963 127.7271245  108.72384895 119.25775732 113.20585803",
    sep=" ",
)


tv_standing_spiro = np.fromstring(
    "2.14765981 2.59810108 2.04987985 2.12560338 2.07429475 2.02826924 1.90354675 1.66025174 1.82820472 2.14254895",
    sep=" ",
)

In [23]:
import numpy as np

# Ensure datasets have the same length
assert len(hexoskin_thorax_depths_standing) == len(tv_standing_spiro)

# Predictor matrix without intercept column
X = hexoskin_thorax_depths_standing[:, np.newaxis]
y = tv_standing_spiro

# Pseudoinverse regression forced through origin (c = 0)
X_pinv = np.linalg.pinv(X)
hexoskin_thorax_slope_standing = np.dot(X_pinv, y)[0]

print(f"Thorax-Only Slope (Standing): {hexoskin_thorax_slope_standing:.8f}")

# Predicted Tidal Volume
tv_standing_thorax_converted = hexoskin_thorax_slope_standing * hexoskin_thorax_depths_standing

# Metrics: MAPE & RMSE
hexoskin_thorax_mape_standing = np.mean(np.abs((y - tv_standing_thorax_converted) / y)) * 100
hexoskin_thorax_rmse_standing = np.sqrt(np.mean((y - tv_standing_thorax_converted) ** 2))

print(f"Thorax-Only Standing MAPE: {hexoskin_thorax_mape_standing:.4f}%")
print(f"Thorax-Only Standing RMSE: {hexoskin_thorax_rmse_standing:.6f} L")

Thorax-Only Slope (Standing): 0.01688919
Thorax-Only Standing MAPE: 10.6188%
Thorax-Only Standing RMSE: 0.258960 L


## 4. Hexoskin Abdomen-Only Calibration (Standing)

In [24]:
import numpy as np


hexoskin_abdomen_depths_standing = np.fromstring(
    "42.34629886 44.48337263 53.79494177 50.2668109  51.21436216 53.29261165 49.76386574 41.48242151 53.11484646 56.48568042",
    sep=" ",
)

tv_standing_spiro = np.fromstring(
    "2.14765981 2.59810108 2.04987985 2.12560338 2.07429475 2.02826924 1.90354675 1.66025174 1.82820472 2.14254895",
    sep=" ",
)

In [25]:
import numpy as np

# Ensure datasets have the same length
assert len(hexoskin_abdomen_depths_standing) == len(tv_standing_spiro)

# Predictor matrix without intercept column
X = hexoskin_abdomen_depths_standing[:, np.newaxis]
y = tv_standing_spiro

# Pseudoinverse regression forced through origin (c = 0)
X_pinv = np.linalg.pinv(X)
hexoskin_abdomen_slope_standing = np.dot(X_pinv, y)[0]

print(f"Abdomen-Only Slope (Standing): {hexoskin_abdomen_slope_standing:.8f}")

# Predicted Tidal Volume
tv_standing_abdomen_converted = hexoskin_abdomen_slope_standing * hexoskin_abdomen_depths_standing

# Metrics: MAPE & RMSE
hexoskin_abdomen_mape_standing = np.mean(np.abs((y - tv_standing_abdomen_converted) / y)) * 100
hexoskin_abdomen_rmse_standing = np.sqrt(np.mean((y - tv_standing_abdomen_converted) ** 2))

print(f"Abdomen-Only Standing MAPE: {hexoskin_abdomen_mape_standing:.4f}%")
print(f"Abdomen-Only Standing RMSE: {hexoskin_abdomen_rmse_standing:.6f} L")

Abdomen-Only Slope (Standing): 0.04100959
Abdomen-Only Standing MAPE: 10.5508%
Abdomen-Only Standing RMSE: 0.315438 L


# Save the output

In [26]:
import pandas as pd

data = {
    "starting_time": [starting_time],
    "shift": [shift],
    "vuams_slope_supine": [vuams_slope_supine],
    "vuams_mape_supine": [vuams_mape_supine],
    "vuams_rmse_supine": [vuams_rmse_supine],

    "hexoskin_thorax_slope_supine": [hexoskin_thorax_slope_supine],
    "hexoskin_thorax_mape_supine": [hexoskin_thorax_mape_supine],
    "hexoskin_thorax_rmse_supine": [hexoskin_thorax_rmse_supine],

    "twodegrees_slope_thorax_supine": [twodegrees_slope_thorax_supine],
    "twodegrees_slope_abdomen_supine": [twodegrees_slope_abdomen_supine],
    "twodegrees_mape_supine": [twodegrees_mape_supine],
    "twodegrees_rmse_supine": [twodegrees_rmse_supine],

    "hexoskin_abdomen_slope_supine": [hexoskin_abdomen_slope_supine],
    "hexoskin_abdomen_mape_supine": [hexoskin_abdomen_mape_supine],
    "hexoskin_abdomen_rmse_supine": [hexoskin_abdomen_rmse_supine],

    # SITTING
    "vuams_slope_sitting": [vuams_slope_sitting],
    "vuams_mape_sitting": [vuams_mape_sitting],
    "vuams_rmse_sitting": [vuams_rmse_sitting],

    "hexoskin_thorax_slope_sitting": [hexoskin_thorax_slope_sitting],
    "hexoskin_thorax_mape_sitting": [hexoskin_thorax_mape_sitting],
    "hexoskin_thorax_rmse_sitting": [hexoskin_thorax_rmse_sitting],

    "twodegrees_slope_thorax_sitting": [twodegrees_slope_thorax_sitting],
    "twodegrees_slope_abdomen_sitting": [twodegrees_slope_abdomen_sitting],
    "twodegrees_mape_sitting": [twodegrees_mape_sitting],
    "twodegrees_rmse_sitting": [twodegrees_rmse_sitting],

    "hexoskin_abdomen_slope_sitting": [hexoskin_abdomen_slope_sitting],
    "hexoskin_abdomen_mape_sitting": [hexoskin_abdomen_mape_sitting],
    "hexoskin_abdomen_rmse_sitting": [hexoskin_abdomen_rmse_sitting],

    # STANDING
    "vuams_slope_standing": [vuams_slope_standing],
    "vuams_mape_standing": [vuams_mape_standing],
    "vuams_rmse_standing": [vuams_rmse_standing],

    "hexoskin_thorax_slope_standing": [hexoskin_thorax_slope_standing],
    "hexoskin_thorax_mape_standing": [hexoskin_thorax_mape_standing],
    "hexoskin_thorax_rmse_standing": [hexoskin_thorax_rmse_standing],

    "twodegrees_slope_thorax_standing": [twodegrees_slope_thorax_standing],
    "twodegrees_slope_abdomen_standing": [twodegrees_slope_abdomen_standing],
    "twodegrees_mape_standing": [twodegrees_mape_standing],
    "twodegrees_rmse_standing": [twodegrees_rmse_standing],

    "hexoskin_abdomen_slope_standing": [hexoskin_abdomen_slope_standing],
    "hexoskin_abdomen_mape_standing": [hexoskin_abdomen_mape_standing],
    "hexoskin_abdomen_rmse_standing": [hexoskin_abdomen_rmse_standing],
}

df = pd.DataFrame(data)

df.to_excel(f"calibration_parameters_{ppid}.xlsx", index=False)